# Best-fit $\chi^2$ summary

Read the global best-fit $\chi^2$ and active-bin count reported by PROfit for every NuWro, open-data, and Asimov fit. We separate $\chi^2_{\rm tot}=\chi^2_{\rm data}+\chi^2_{\rm penalty}$ so spectrum comparisons use the consistently defined data term. The reduced value is $\chi^2_{\rm data}/N_{\rm dof}$, where $N_{\rm dof}=N_{\rm bins}-N_{\rm fitted}$. Uniform parameters contribute zero pull penalty; other Gaussian-constrained nuisance parameters in the same fit still contribute. Missing or unfinished fits are retained as `NaN` so gaps are visible.

In [ ]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

OUTPUT_ROOT = Path("/nevis/riverside/data/epelaez/ma_zexp/1mu1p_sel")

SUITES = {
    "NuWro": "zexp_prior_fits",
    "Open data": "zexp_prior_fits_opendata",
    "Asimov": "zexp_prior_fits_asimov",
}
FIT_ORDER = [
    "ma", "ma_uniform", "ma_no_axff", "lqcd_k6", "minerva_k6",
    "minerva_k7", "minerva_k8", "minerva_lqcd_k6",
    "minerva_k6_uniform", "minerva_k6_nuisance"
]
N_FITTED = {
    "ma": 4, "ma_uniform": 3, "ma_no_axff": 3,
    "lqcd_k6": 2, "minerva_k6": 2, "minerva_k7": 3,
    "minerva_k8": 4, "minerva_lqcd_k6": 2,
    "minerva_k6_uniform": 2, "minerva_k6_nuisance": 4,
}
UNIFORM_PARAMETERS = {
    "ma_uniform": {"MACCQE"},
    "minerva_k6_uniform": {"FAzexpMinervaK6PCA1", "FAzexpMinervaK6PCA2"},
}
CHI2_PATTERN = re.compile(
    r"Global Best Fit chi(?:\^?2|²):\s*"
    r"([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)"
)
NBINS_PATTERN = re.compile(r"/\s*(\d+)\s+reco bins")
PARAM_PATTERN = re.compile(
    r"main \|\|\s+(\w+)\s+:\s*"
    r"([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)"
)

In [ ]:
def read_fit_stats(log_file):
    """Return total chi2, active-bin count, and best-fit parameters."""
    if not log_file.is_file():
        return float("nan"), float("nan"), {}
    text = log_file.read_text(errors="replace")
    chi2_matches = CHI2_PATTERN.findall(text)
    nbins_matches = NBINS_PATTERN.findall(text)
    chi2 = float(chi2_matches[-1]) if chi2_matches else float("nan")
    nbins = int(nbins_matches[-1]) if nbins_matches else float("nan")
    final_block = text.rsplit("Global Best Fit chi^2:", 1)[-1]
    parameters = {name: float(value) for name, value in PARAM_PATTERN.findall(final_block)}
    return chi2, nbins, parameters

def pull_penalty(fit, parameters):
    """Calculate the Gaussian pull term; uniform parameters contribute zero."""
    uniform = UNIFORM_PARAMETERS.get(fit, set())
    penalty = 0.0
    for name, value in parameters.items():
        if name in uniform:
            continue
        penalty += value ** 2
    return penalty

records = []
for sample, suite in SUITES.items():
    for fit in FIT_ORDER:
        log_file = OUTPUT_ROOT / suite / fit / "profile.log"
        chi2_tot, nbins, parameters = read_fit_stats(log_file)
        chi2_penalty = pull_penalty(fit, parameters)
        chi2_data = chi2_tot - chi2_penalty
        n_fitted = N_FITTED[fit]
        ndof = nbins - n_fitted
        records.append({
            "fit": fit,
            "sample": sample,
            "chi2_data": chi2_data,
            "chi2_penalty": chi2_penalty,
            "chi2_tot": chi2_tot,
            "nbins": nbins,
            "n_fitted": n_fitted,
            "ndof": ndof,
            "chi2_data_per_ndof": chi2_data / ndof,
            "profile_log": str(log_file),
        })

chi2_long = pd.DataFrame(records)

In [ ]:
shared_columns = ["nbins", "n_fitted", "ndof"]
shared = chi2_long.groupby("fit")[shared_columns].first().reindex(FIT_ORDER)
shared.columns = pd.MultiIndex.from_product(
    [shared_columns, [""]], names=["quantity", "Data sample"]
)
sample_results = chi2_long.pivot(
    index="fit", columns="sample",
    values=["chi2_data", "chi2_penalty", "chi2_tot", "chi2_data_per_ndof"],
).reindex(index=FIT_ORDER)
sample_results = sample_results.reindex(
    columns=pd.MultiIndex.from_product(
        [["chi2_data", "chi2_penalty", "chi2_tot", "chi2_data_per_ndof"], list(SUITES)],
        names=["quantity", "Data sample"],
    )
)
chi2_table = pd.concat([shared, sample_results], axis=1)
chi2_table.index.name = "Fit"
display(chi2_table.style.format({
    "chi2_data": "{:.4f}",
    "chi2_penalty": "{:.4f}",
    "chi2_tot": "{:.4f}",
    "nbins": "{:.0f}",
    "n_fitted": "{:.0f}",
    "ndof": "{:.0f}",
    "chi2_data_per_ndof": "{:.4f}",
}, na_rep="not available"))

The long-form table below includes the source log path, which is useful for checking any missing entry.

In [ ]:
display(chi2_long)

In [ ]:
LATEX_FIT_ORDER = [
    "ma_no_axff", "ma_uniform", "minerva_k8", "minerva_k7",
    "minerva_k6", "lqcd_k6", "minerva_lqcd_k6",
    "minerva_k6_nuisance", "minerva_k6_uniform",
]
LATEX_FIT_LABELS = {
    "ma_no_axff": r"$M_A$ Gaussian prior",
    "ma_uniform": r"$M_A$ uniform prior",
    "minerva_k8": r"MINERvA, $k_{\max}=8$",
    "minerva_k7": r"MINERvA, $k_{\max}=7$",
    "minerva_k6": r"MINERvA, $k_{\max}=6$",
    "lqcd_k6": r"LQCD, $k_{\max}=6$",
    "minerva_lqcd_k6": r"MINERvA+LQCD, $k_{\max}=6$",
    "minerva_k6_nuisance": r"MINERvA w/ nuisance, $k_{\max}=6$",
    "minerva_k6_uniform": r"Uniform prior, $k_{\max}=6$",
}

nuwro_results = chi2_long.query("sample == 'NuWro'").set_index("fit")
latex_lines = [
    r"\begin{ruledtabular}",
    r"    \begin{tabular}{lccc}",
    r"        Fit & $\chi^2_{\mathrm{data}}$ & $\chi^2_{\mathrm{data}}/\nu$ & $\chi^2_{\mathrm{pull}}$ \\",
    r"        \hline",
]
for fit in LATEX_FIT_ORDER:
    row = nuwro_results.loc[fit]
    latex_lines.append(
        f"        {LATEX_FIT_LABELS[fit]} & {row.chi2_data:.3f} & "
        f"{row.chi2_data_per_ndof:.3f} & {row.chi2_penalty:.3f} \\\\"
    )
latex_lines.extend([r"    \end{tabular}", r"\end{ruledtabular}"])
print("\n".join(latex_lines))